In [1]:
# !pip install numpy 
# !pip install pandas 
# !pip install sqlalchemy
# !pip install pymysql 

In [2]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

In [3]:
# Note the empty space between root: and @
DATABASE_URL = "mysql+pymysql://root:@localhost:3306/kyc_db"

# engine = create_engine(DATABASE_URL)
# DATABASE_URL = "jdbc:mysql://localhost:3306/kyc_db"
engine = create_engine(DATABASE_URL)

sql_query_client = text("""SELECT * FROM client;""")
sql_query_cases = text("""SELECT * FROM onboarding_case;""")
sql_query_documents = text("""SELECT * FROM document;""")
sql_query_addresses = text("""SELECT * FROM client_address;""")
sql_query_risk = text("""SELECT * FROM risk_classification;""")


with engine.connect() as conn:
    df_clients = pd.read_sql(sql_query_client, con=conn)
    df_cases = pd.read_sql(sql_query_cases, con=conn)
    df_documents = pd.read_sql(sql_query_documents,con=conn )
    df_addresses = pd.read_sql(sql_query_addresses,con=conn )
    df_risk = pd.read_sql(sql_query_risk, con=conn)

In [4]:
# ---------------------------------------------------------
# 2. FEATURE ENGINEERING & AGGREGATIONS
# ---------------------------------------------------------
# Convert dates
df_clients["date_of_birth"] = pd.to_datetime(df_clients["date_of_birth"])
df_documents["expiry_date"] = pd.to_datetime(df_documents["expiry_date"])
today = pd.to_datetime("today")

# Compute Client Age
df_clients["age"] = (today - df_clients["date_of_birth"]).dt.days // 365

# Aggregate Document-level stats per case
doc_aggregations = df_documents.groupby("case_id").agg(
    total_docs_submitted=("doc_id", "count"),
    verified_docs_count=("verified_flag", "sum"),
    expired_docs_count=("expiry_date", lambda dates: (dates < today).sum()),
    has_unverified_docs=("verified_flag", lambda flags: int((~flags).any()))
).reset_index()

# Check cross-border address vs tax residency
# Drop columns from a previous run of this cell so the merge stays idempotent
df_clients = df_clients.drop(columns=["address_country", "is_cross_border"], errors="ignore")

df_address_primary = df_addresses[df_addresses["address_type"] == "REGISTERED"].drop_duplicates("client_id")
df_clients = df_clients.merge(
    df_address_primary[["client_id", "country"]].rename(columns={"country": "address_country"}),
    on="client_id",
    how="left"
)
df_clients["is_cross_border"] = (df_clients["tax_residency"] != df_clients["address_country"]).astype(int)



In [5]:
# ---------------------------------------------------------
# 3. MERGE INTO A SINGLE ANALYTICAL / ML DATAFRAME
# ---------------------------------------------------------
# Base merge: case -> client
df_analysis = df_cases.merge(
    df_clients,
    on="client_id",
    how="inner",
    suffixes=("_case", "_client")
)

# Merge document aggregations
df_analysis = df_analysis.merge(
    doc_aggregations,
    on="case_id",
    how="left"
)

# Fill any missing doc aggregations (cases with 0 docs)
df_analysis["total_docs_submitted"] = df_analysis["total_docs_submitted"].fillna(0).astype(int)
df_analysis["verified_docs_count"] = df_analysis["verified_docs_count"].fillna(0).astype(int)
df_analysis["expired_docs_count"] = df_analysis["expired_docs_count"].fillna(0).astype(int)
df_analysis["has_unverified_docs"] = df_analysis["has_unverified_docs"].fillna(1).astype(int)

# Target Variable: 1 = APPROVED (Active), 0 = REJECTED
df_analysis["is_approved"] = (df_analysis["status"] == "ACTIVE").astype(int)



In [6]:
# ---------------------------------------------------------
# 4. SELECT CLEAN ML/ANALYTICS FEATURE SET
# ---------------------------------------------------------
feature_columns = [
    # Demographics & Entity Info
    "client_type",
    "nationality",
    "jurisdiction_risk",
    "age",
    "annual_income_band",
    "main_source_of_funds",
    "is_pep",
    "adverse_media_hits",
    "is_cross_border",
    # Case & Application Scope
    "product_type",
    # Compliance & Verification Levers
    "total_docs_submitted",
    "verified_docs_count",
    "expired_docs_count",
    "has_unverified_docs",
    # Target
    "is_approved"
]

df_ml = df_analysis[feature_columns].copy()



In [7]:
# ---------------------------------------------------------
# 5. QUICK DATA AUDIT & INSPECTION
# ---------------------------------------------------------
print(f"Shape of flattened dataset: {df_ml.shape}")
print("\n--- CLASS BALANCE (Target: is_approved) ---")
print(df_ml["is_approved"].value_counts(normalize=True).round(3))

print("\n--- SAMPLE ROWS ---")
print(df_ml.head(3).T)

print("\n--- APPROVAL RATE BY JURISDICTION RISK ---")
print(df_ml.groupby("jurisdiction_risk")["is_approved"].mean().round(3))

Shape of flattened dataset: (10000, 15)

--- CLASS BALANCE (Target: is_approved) ---
is_approved
1    0.874
0    0.126
Name: proportion, dtype: float64

--- SAMPLE ROWS ---
                                      0                     1  \
client_type                  INDIVIDUAL             CORPORATE   
nationality                          AU                    YE   
jurisdiction_risk                   LOW                  HIGH   
age                                  70                    18   
annual_income_band              50-100K                25-50K   
main_source_of_funds  Employment Income   Business Operations   
is_pep                                0                     0   
adverse_media_hits                    0                     0   
is_cross_border                       0                     0   
product_type             AUTO_FINANCING  LIQUIDITY_MANAGEMENT   
total_docs_submitted                  2                     2   
verified_docs_count                   0        

# Model

In [9]:
# !pip install lightgbm scikit-learn matplotlib seaborn


In [11]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

# Assume df_ready contains your engineered features from the previous step
# (e.g., client_type, nationality, age, doc_verification_ratio, is_approved, etc.)
df_ready = df_ml

target = "is_approved"
categorical_features = [
    "client_type",
    "nationality",
    "jurisdiction_risk",
    "product_type",
]

# Ensure categorical columns use the pandas 'category' dtype
for col in categorical_features:
    df_ready[col] = df_ready[col].astype("category")

X = df_ready.drop(columns=[target])
y = df_ready[target]

# 80/20 Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Initialize LightGBM Classifier
clf = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",  # Handles imbalanced approvals/rejections
    random_state=42,
)

# Train with early stopping against the validation set
clf.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_names=["Train", "Validation"],
    eval_metric=["binary_logloss", "auc"],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
)

# Generate Predictions and Probabilities
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

c:\HSBC\Neueda\Project\KYC-Client-Onboarding\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: annual_income_band: str, main_source_of_funds: str

In [ ]:
# 1. Standard Classification Metrics
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=["REJECTED", "APPROVED"]))

roc_auc = roc_auc_score(y_test, y_proba)
loss = log_loss(y_test, y_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"Log Loss:      {loss:.4f}")

# 2. Confusion Matrix & ROC Curve Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["REJECTED", "APPROVED"],
    yticklabels=["REJECTED", "APPROVED"],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix")
axes[0].set_ylabel("Actual Status")
axes[0].set_xlabel("Predicted Status")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", linestyle="--")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

# 3. Native LightGBM Feature Importance
lgb.plot_importance(clf, max_num_features=10, importance_type="gain", figsize=(8, 5))
plt.title("Top 10 Features by Information Gain")
plt.show()